Install Swabian timetagger(https://pypi.org/project/Swabian-TimeTagger/), 
Thorlabs elliptec(https://pypi.org/project/thorlabs-elliptec/)

If the power supplies are disconnected from Thorlabs Ellx series, define address again with disconnecting each rotators.

In [1]:
import numpy as np
import time as time
# import pyvisa as visa
import os
# import serial.tools.list_ports
from TimeTagger import *
from thorlabs_elliptec import *
from EllxBus import *
from Ellx_addressing import *

In [2]:
ports = EllxAddressing.find_ell_ports()
print(ports)

Found: COM3 - USB Serial Port(COM3) - 1027:24597 - DK0IVVHSA
Found: COM6 - USB Serial Port(COM6) - 1027:24597 - DP06UFXVA
['COM3', 'COM6']


Before connect ELL bus to USB port, check which port is connected in which Buses.

first one should be connected to tomography waveplates set (3 addresses)

In [3]:
Tomo_set = EllxAddressing(ports[0])
Source_set = EllxAddressing(ports[1])
Idler_QWP = ELLx(x = 14, serial_port = ports[1], device_id = 1)
Idler_HWP = ELLx(x = 14, serial_port = Idler_QWP, device_id = 2)
Comp_QWP_out = ELLx(x = 14, serial_port = ports[0], device_id = 3)
Comp_HWP = ELLx(x = 14, serial_port = Comp_QWP_out, device_id = 2)
Comp_QWP_in = ELLx(x = 14, serial_port = Comp_QWP_out, device_id = 1)
# Tomo_QWP = ELLx(x = 14, serial_port = ports[0], device_id = 1)
# Tomo_HWP = ELLx(x = 14, serial_port = Tomo_QWP, device_id = 2)

In [ ]:
## Swebian time tagger connection and channel setting - remote control from TimeTagger mini-pc
Tagger = createTimeTaggerNetwork('172.30.63.109')

Input_channels = [1, 2]

T_delay = 51                                     # delay in ns

Coinc_window = 1                                # in ns
Coinc_window = Coinc_window * 1e3               # in ps

Tagger.setInputDelay(Input_channels[1], (T_delay) * 1000)
# set appropriate delay to channel, depending on position of transmitted correlation peak

Coinc_bandwidth = 1                            # in s  

Counter_bandwidth = 1                           # in s

In [5]:
# Histo = Histogram(Tagger, Input_channels[0], Input_channels[1], Coinc_window, 10)

# Histo.clear()
# Histo.startFor(int(Coinc_bandwidth * 1e12), clear = True)
# Histo.waitUntilFinished()
# aaaa = Histo.getData()[:]
# print(aaaa[0])

In [6]:
# counter = Counter(Tagger, Input_channels, Counter_bandwidth * 1e12 , int(Coinc_bandwidth/Counter_bandwidth))

# counter = Countrate(Tagger, [1,2])
# counter.startFor(1, clear = True)
# # counter.waitUntilFinished
# Single_result = np.zeros([2,10])
# Single_result[0,:] = counter.getData()[0]
# Single_result[1,:] = counter.getData()[1]
# # print(Single_result)
# print(counter.getData())

Projection measurement are ordered


Idler: HWP - QWP - PBS transmission(H pol. state)


Tomography(Signal): HWP - QWP - PBS transmission(H pol. state)


And compensator set,

Compensator: QWP1 - HWP - QWP2 - Fiber

In [2]:
## Calibrated waveplate angle set [HWP, QWP]
Compensator_angle = np.zeros([6,3])
## [QWP_in, HWP, QWP_out]
Compensator_angle[0,:] = [219.3, 44.6, 186.1]             # H
Compensator_angle[1,:] = [219.3, 44.6+45, 186.1]             # V
Compensator_angle[2,:] = [219.3-45, 44.6+22.5, 186.1]            # D
Compensator_angle[3,:] = [219.3-45, 44.6-22.5, 186.1]            # A
Compensator_angle[4,:] = [219.3+45, 44.6, 186.1]            # R
Compensator_angle[5,:] = [219.3-45, 44.6, 186.1]            # L
Idler_angle = np.zeros([6,2])
## [HWP, QWP]
Idler_angle[0,:] = [184.89, 147.23]                     # H
Idler_angle[1,:] = [184.89 + 45.7, 148.43]              # V
Idler_angle[2,:] = [184.89 + 20.8, 143.45]              # D
Idler_angle[3,:] = [184.89 - 20.8, 151]                 # A
Idler_angle[4,:] = [184.89, 147.23 - 47.3]              # R
Idler_angle[5,:] = [184.89, 147.23 + 48.24]             # L
Tomo_angle = np.zeros([6,2])
## [HWP, QWP]
Tomo_angle[0,:] = [134.0, 39.82]                     # H
Tomo_angle[1,:] = [134.33 + 44.3, 39.82]              # V
Tomo_angle[2,:] = [134.0 + 68.3, 41.82]              # D
Tomo_angle[3,:] = [134.0 + 20.7, 36.82]                 # A
Tomo_angle[4,:] = [138.8, 39.82 - 131.8]              # R
Tomo_angle[5,:] = [137.0, 39.82 + 130.65]             # L

In [3]:
np.savetxt("Compensator_angle_20260430.csv", Compensator_angle, delimiter = ',')

In [8]:
## Tomography running
# Labels = ['Signal', 'Idler', 'Coincidence']
counter = Counter(Tagger, Input_channels, Counter_bandwidth * 1e12 , int(Coinc_bandwidth/Counter_bandwidth))
Histo = Histogram(Tagger, Input_channels[0], Input_channels[1], Coinc_window, 1)

Tomo_result = np.zeros([36,3])
for ii in range(6):
    Comp_QWP_in.move_absolute(Compensator_angle[ii, 0])
    Comp_QWP_in.wait(raise_errors=True)
    Comp_HWP.move_absolute(Compensator_angle[ii, 1])
    Comp_HWP.wait(raise_errors=True)
    Comp_QWP_out.move_absolute(Compensator_angle[ii, 2])
    Comp_QWP_out.wait(raise_errors=True)
    for iii in range(6):
        Idler_HWP.move_absolute(Idler_angle[iii, 0])
        Idler_HWP.wait(raise_errors=True)
        Idler_QWP.move_absolute(Idler_angle[iii, 1])
        Idler_QWP.wait(raise_errors=True)
        time.sleep(5)
        Histo.clear()
        Histo.startFor(int(Coinc_bandwidth * 1e12), clear = True)
        Histo.waitUntilFinished()
        Tomo_result[(ii * 6) + iii, 2] = int(Histo.getData()[0])
        Tomo_result[(ii * 6) + iii, 0] = ii + 1
        Tomo_result[(ii * 6) + iii, 1] = iii + 1


Set Compensator as H polarization

In [9]:
# Comp_QWP_in.move_absolute(Compensator_angle[0, 0])
# Comp_QWP_in.wait(raise_errors=True)
# Comp_HWP.move_absolute(Compensator_angle[0, 1])
# Comp_HWP.wait(raise_errors=True)
# Comp_QWP_out.move_absolute(Compensator_angle[0, 2])
# Comp_QWP_out.wait(raise_errors=True)

Only H polarization set from compensator waveplate group

In [10]:
# Labels = ['Signal', 'Idler', 'Coincidence']
# counter = Counter(Tagger, Input_channels, Counter_bandwidth * 1e12 , int(Coinc_bandwidth/Counter_bandwidth))
# Histo = Histogram(Tagger, Input_channels[0], Input_channels[1], Coinc_window, 2)

# Tomo_result = np.zeros([36,3])
# for ii in range(6):
#     Idler_HWP.move_absolute(Idler_angle[ii, 0])
#     Idler_HWP.wait(raise_errors=True)
#     Idler_QWP.move_absolute(Idler_angle[ii, 1])
#     Idler_QWP.wait(raise_errors=True)
#     for iii in range(6):
#         Tomo_HWP.move_absolute(Tomo_angle[iii, 0])
#         Tomo_HWP.wait(raise_errors=True)
#         Tomo_QWP.move_absolute(Tomo_angle[iii, 1])
#         Tomo_QWP.wait(raise_errors=True)
#         time.sleep(0.5)
#         Histo.clear()
#         Histo.startFor(int(Coinc_bandwidth * 1e12), clear = True)
#         Histo.waitUntilFinished()
#         Tomo_result[(ii * 6) + iii, 2] = Histo.getData()[0]
#         Tomo_result[(ii * 6) + iii, 0] = ii + 1
#         Tomo_result[(ii * 6) + iii, 1] = iii + 1


Changing compensator setup as H V D A R L (36 * 6 times measurement)

In [11]:
# Labels = ['Signal', 'Idler', 'Coincidence']
# counter = Counter(Tagger, Input_channels, Counter_bandwidth * 1e12 , int(Coinc_bandwidth/Counter_bandwidth))
# Histo = Histogram(Tagger, Input_channels[0], Input_channels[1], Coinc_window, 2)

# Tomo_result = np.zeros([36*6, 4])
# for kk in range(6):
#     Comp_QWP_in.move_absolute(Compensator_angle[kk, 0])
#     Comp_QWP_in.wait(raise_errors=True)
#     Comp_HWP.move_absolute(Compensator_angle[kk, 1])
#     Comp_HWP.wait(raise_errors=True)
#     Comp_QWP_out.move_absolute(Compensator_angle[kk, 2])
#     Comp_QWP_out.wait(raise_errors=True)
#     for ii in range(6):
#         Idler_HWP.move_absolute(Idler_angle[ii, 0])
#         Idler_HWP.wait(raise_errors=True)
#         Idler_QWP.move_absolute(Idler_angle[ii, 1])
#         Idler_QWP.wait(raise_errors=True)
#         for iii in range(6):
#             Tomo_HWP.move_absolute(Tomo_angle[iii, 0])
#             Tomo_HWP.wait(raise_errors=True)
#             Tomo_QWP.move_absolute(Tomo_angle[iii, 1])
#             Tomo_QWP.wait(raise_errors=True)
#             time.sleep(0.5)
#             Histo.clear()
#             Histo.startFor(int(Coinc_bandwidth * 1e12), clear = True)
#             Histo.waitUntilFinished()
#             Tomo_result[(kk * 6) + (ii * 6) + iii, 3] = int(Histo.getData()[0])
#             Tomo_result[(kk * 6) + (ii * 6) + iii, 0] = kk + 1
#             Tomo_result[(kk * 6) + (ii * 6) + iii, 1] = ii + 1
#             Tomo_result[(kk * 6) + (ii * 6) + iii, 2] = iii + 1


In [12]:
# Single_result = np.zeros([2,int(Coinc_bandwidth/Counter_bandwidth)])
# Single_result[0,:] = counter.getData()[0][:]
# Single_result[1,:] = counter.getData()[1][:]
# print(Single_result)

In [13]:
np.savetxt("AAPT_test_1sec_20260430.csv", Tomo_result, delimiter = ',')
# np.savetxt("AAPT_test_single_counts_20260416.csv", Single_result, delimiter = ',')

In [14]:
Idler_QWP.close()
Idler_HWP.close()
Comp_QWP_out.close()
Comp_HWP.close()
Comp_QWP_in.close()

In [15]:
# Idler_state = 3
# Comp_state = 2

# Idler_HWP.move_absolute(Idler_angle[Idler_state,0])
# Idler_QWP.move_absolute(Idler_angle[Idler_state,1])
# Comp_QWP_in.move_absolute(Compensator_angle[Comp_state,0])
# Comp_HWP.move_absolute(Compensator_angle[Comp_state,1])
# Comp_QWP_out.move_absolute(Compensator_angle[Comp_state,2])